In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path(r"D:\Study\study\Project\2nd Project")
RAW_PATH = PROJECT_DIR / "data" / "row" / "data.json"

print("Python 실행 경로:", sys.executable)
print("프로젝트 폴더 존재:", PROJECT_DIR.is_dir())
print("원본 파일 존재:", RAW_PATH.is_file())

if RAW_PATH.is_file():
    print(
        "원본 크기:",
        round(RAW_PATH.stat().st_size / (1024 ** 3), 2),
        "GiB"
    )

Python 실행 경로: c:\Users\s2sad\miniconda3\envs\agent\python.exe
프로젝트 폴더 존재: True
원본 파일 존재: True
원본 크기: 11.88 GiB


In [2]:
# 작업 결과를 저장할 폴더
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("전처리 데이터 저장 위치:", PROCESSED_DIR)
print("검사·분석 보고서 저장 위치:", REPORT_DIR)
print("원본 파일은 변경하지 않았습니다.")

전처리 데이터 저장 위치: D:\Study\study\Project\2nd Project\data\processed
검사·분석 보고서 저장 위치: D:\Study\study\Project\2nd Project\reports
원본 파일은 변경하지 않았습니다.


In [3]:
import json
from collections import Counter

sample_records = []

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"{line_number}번째 줄을 JSON으로 읽지 못했습니다."
            ) from e

        if not isinstance(record, dict):
            raise TypeError(
                f"{line_number}번째 줄이 이벤트 객체가 아닙니다."
            )

        sample_records.append(record)

        if len(sample_records) == 5:
            break

if not sample_records:
    raise ValueError("읽을 수 있는 이벤트가 없습니다.")

# 5개 기록에서 발견한 모든 항목
sample_fields = sorted({
    field
    for record in sample_records
    for field in record
})

print("읽은 이벤트 수:", len(sample_records))
print("확인한 항목 수:", len(sample_fields))

print("\n항목별 자료형 — 앞 5개 기준")

for field in sample_fields:
    types = Counter(
        type(record[field]).__name__
        if field in record else "항목 없음"
        for record in sample_records
    )

    print(f"{field}: {dict(types)}")

읽은 이벤트 수: 5
확인한 항목 수: 18

항목별 자료형 — 앞 5개 기준
artist: {'str': 5}
auth: {'str': 5}
firstName: {'str': 5}
gender: {'str': 5}
itemInSession: {'int': 5}
lastName: {'str': 5}
length: {'float': 5}
level: {'str': 5}
location: {'str': 5}
method: {'str': 5}
page: {'str': 5}
registration: {'int': 5}
sessionId: {'int': 5}
song: {'str': 5}
status: {'int': 5}
ts: {'int': 5}
userAgent: {'str': 5}
userId: {'str': 5}


In [4]:
# 문자열이 들어와야 하는 항목
STRING_FIELDS = {
    "artist",
    "auth",
    "firstName",
    "gender",
    "lastName",
    "level",
    "location",
    "method",
    "page",
    "song",
    "userAgent",
    "userId",
}

# 정수가 들어와야 하는 항목
INTEGER_FIELDS = {
    "itemInSession",
    "registration",
    "sessionId",
    "status",
    "ts",
}

# 정수 또는 소수가 들어올 수 있는 항목
NUMBER_FIELDS = {"length"}

# 이번 분석의 고객 구분·시간·행동·세션 집계에 필요한 항목
REQUIRED_FIELDS = {
    "userId",
    "ts",
    "page",
    "sessionId",
}

EXPECTED_FIELDS = (
    STRING_FIELDS | INTEGER_FIELDS | NUMBER_FIELDS
)

# 앞에서 확인한 항목과 검사 기준이 일치하는지 확인
assert EXPECTED_FIELDS == set(sample_fields)

print("검사 기준 설정 완료")
print("전체 검사 대상 항목:", len(EXPECTED_FIELDS))
print("필수 항목:", ", ".join(sorted(REQUIRED_FIELDS)))

검사 기준 설정 완료
전체 검사 대상 항목: 18
필수 항목: page, sessionId, ts, userId


In [5]:
import json
import math
import time
from collections import Counter

# 항목별 검사 결과
field_checks = {
    field: Counter()
    for field in sorted(EXPECTED_FIELDS)
}

# 파일 구조 검사 결과
structure_checks = Counter()

# 예상하지 못한 항목
unexpected_fields = Counter()

# 문제가 발생한 줄 번호만 일부 기록
# 이름이나 지역 등 원본 개인정보는 기록하지 않습니다.
issue_examples = {}


def remember_issue(issue_name, line_number):
    examples = issue_examples.setdefault(issue_name, [])

    if len(examples) < 5:
        examples.append(line_number)


def valid_number(value):
    # True/False는 숫자로 인정하지 않습니다.
    if isinstance(value, bool):
        return False

    if isinstance(value, int):
        return True

    if isinstance(value, float):
        return math.isfinite(value)

    return False


started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line_number, line in enumerate(f, start=1):
        structure_checks["전체 줄 수"] += 1

        if not line.strip():
            structure_checks["빈 줄"] += 1
            continue

        try:
            record = json.loads(line)
        except json.JSONDecodeError:
            structure_checks["JSON 읽기 실패"] += 1
            remember_issue("JSON 읽기 실패", line_number)
            continue

        if not isinstance(record, dict):
            structure_checks["객체가 아닌 기록"] += 1
            remember_issue("객체가 아닌 기록", line_number)
            continue

        structure_checks["읽은 이벤트 수"] += 1

        for field in record.keys() - EXPECTED_FIELDS:
            unexpected_fields[field] += 1

        for field in sorted(EXPECTED_FIELDS):
            result = field_checks[field]

            # 1. 항목 자체가 없는 경우
            if field not in record:
                result["항목 없음"] += 1
                remember_issue(
                    f"{field}: 항목 없음", line_number
                )
                continue

            value = record[field]

            # 실제 자료형도 기록
            result[f"자료형_{type(value).__name__}"] += 1

            # 2. 값이 null인 경우
            if value is None:
                result["null"] += 1
                remember_issue(
                    f"{field}: null", line_number
                )
                continue

            # 3. 문자열의 빈값과 앞뒤 공백
            if isinstance(value, str):
                stripped = value.strip()

                if stripped == "":
                    result["빈 문자열 또는 공백만 있음"] += 1
                    remember_issue(
                        f"{field}: 빈 문자열", line_number
                    )

                if value != stripped:
                    result["앞뒤 공백 있음"] += 1
                    remember_issue(
                        f"{field}: 앞뒤 공백", line_number
                    )

            # 4. 예상 자료형 검사
            if field in STRING_FIELDS:
                type_ok = isinstance(value, str)

            elif field in INTEGER_FIELDS:
                type_ok = (
                    isinstance(value, int)
                    and not isinstance(value, bool)
                )

            else:
                type_ok = valid_number(value)

            if not type_ok:
                result["예상 자료형과 다름"] += 1
                remember_issue(
                    f"{field}: 자료형 불일치", line_number
                )

        # 500만 이벤트마다 진행 상황 표시
        count = structure_checks["읽은 이벤트 수"]

        if count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at

            print(
                f"{count:,}건 검사 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

elapsed = time.perf_counter() - started_at

# 결과를 파일로 저장
report = {
    "원본 경로": str(RAW_PATH),
    "구조 검사": dict(structure_checks),
    "항목별 검사": {
        field: dict(result)
        for field, result in field_checks.items()
    },
    "예상하지 못한 항목": dict(unexpected_fields),
    "확인용 줄 번호": issue_examples,
}

report_path = REPORT_DIR / "01_원본_구조_자료형_검사.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        report,
        f,
        ensure_ascii=False,
        indent=2
    )

# 화면에는 짧게 출력
print("\n===== 전체 검사 완료 =====")

for name in [
    "전체 줄 수",
    "읽은 이벤트 수",
    "빈 줄",
    "JSON 읽기 실패",
    "객체가 아닌 기록",
]:
    print(f"{name}: {structure_checks[name]:,}")

print("\n항목별 확인 사항")
print("항목 | 항목 없음 | null | 빈 문자열 | 앞뒤 공백 | 자료형 불일치")

for field, result in field_checks.items():
    print(
        f"{field} | "
        f"{result['항목 없음']:,} | "
        f"{result['null']:,} | "
        f"{result['빈 문자열 또는 공백만 있음']:,} | "
        f"{result['앞뒤 공백 있음']:,} | "
        f"{result['예상 자료형과 다름']:,}"
    )

print("\n예상하지 못한 항목:", dict(unexpected_fields))
print("검사 결과 저장:", report_path)
print(f"실행 시간: {elapsed / 60:.1f}분")
print("원본과 기존 CSV는 변경하지 않았습니다.")

5,000,000건 검사 중 | 4.6분
10,000,000건 검사 중 | 8.9분
15,000,000건 검사 중 | 14.6분
20,000,000건 검사 중 | 20.0분
25,000,000건 검사 중 | 25.1분

===== 전체 검사 완료 =====
전체 줄 수: 26,259,199
읽은 이벤트 수: 26,259,199
빈 줄: 0
JSON 읽기 실패: 0
객체가 아닌 기록: 0

항목별 확인 사항
항목 | 항목 없음 | null | 빈 문자열 | 앞뒤 공백 | 자료형 불일치
artist | 5,408,927 | 0 | 0 | 1,361 | 0
auth | 0 | 0 | 0 | 0 | 0
firstName | 778,479 | 0 | 0 | 0 | 0
gender | 778,479 | 0 | 0 | 0 | 0
itemInSession | 0 | 0 | 0 | 0 | 0
lastName | 778,479 | 0 | 0 | 0 | 0
length | 5,408,927 | 0 | 0 | 0 | 0
level | 0 | 0 | 0 | 0 | 0
location | 778,479 | 0 | 0 | 0 | 0
method | 0 | 0 | 0 | 0 | 0
page | 0 | 0 | 0 | 0 | 0
registration | 778,479 | 0 | 0 | 0 | 0
sessionId | 0 | 0 | 0 | 0 | 0
song | 5,408,927 | 0 | 0 | 15,405 | 0
status | 0 | 0 | 0 | 0 | 0
ts | 0 | 0 | 0 | 0 | 0
userAgent | 778,479 | 0 | 0 | 0 | 0
userId | 0 | 0 | 0 | 0 | 0

예상하지 못한 항목: {}
검사 결과 저장: D:\Study\study\Project\2nd Project\reports\01_원본_구조_자료형_검사.json
실행 시간: 26.5분
원본과 기존 CSV는 변경하지 않았습니다.


In [6]:
import json

report_path = REPORT_DIR / "01_원본_구조_자료형_검사.json"

with report_path.open("r", encoding="utf-8") as f:
    audit_report = json.load(f)

print("항목 | 없음 | null | 빈 문자열 | 앞뒤 공백 | 자료형 불일치")

for field, result in audit_report["항목별 검사"].items():
    print(
        f"{field} | "
        f"{result.get('항목 없음', 0):,} | "
        f"{result.get('null', 0):,} | "
        f"{result.get('빈 문자열 또는 공백만 있음', 0):,} | "
        f"{result.get('앞뒤 공백 있음', 0):,} | "
        f"{result.get('예상 자료형과 다름', 0):,}"
    )

항목 | 없음 | null | 빈 문자열 | 앞뒤 공백 | 자료형 불일치
artist | 5,408,927 | 0 | 0 | 1,361 | 0
auth | 0 | 0 | 0 | 0 | 0
firstName | 778,479 | 0 | 0 | 0 | 0
gender | 778,479 | 0 | 0 | 0 | 0
itemInSession | 0 | 0 | 0 | 0 | 0
lastName | 778,479 | 0 | 0 | 0 | 0
length | 5,408,927 | 0 | 0 | 0 | 0
level | 0 | 0 | 0 | 0 | 0
location | 778,479 | 0 | 0 | 0 | 0
method | 0 | 0 | 0 | 0 | 0
page | 0 | 0 | 0 | 0 | 0
registration | 778,479 | 0 | 0 | 0 | 0
sessionId | 0 | 0 | 0 | 0 | 0
song | 5,408,927 | 0 | 0 | 15,405 | 0
status | 0 | 0 | 0 | 0 | 0
ts | 0 | 0 | 0 | 0 | 0
userAgent | 778,479 | 0 | 0 | 0 | 0
userId | 0 | 0 | 0 | 0 | 0


In [7]:
import json
import time
from collections import Counter, defaultdict

# 곡 정보가 누락된 이벤트 종류와 항목별 건수
music_missing_by_page = defaultdict(Counter)

# 가입 시각이 없는 기록의 인증 상태
registration_missing_by_auth = Counter()

# 가입 시각이 없는 기록의 고객 ID별 건수
registration_missing_by_user = Counter()

# 곡 재생 이벤트에서 곡 정보가 누락되는지 확인
nextsong_missing = Counter()

event_count = 0
started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        event_count += 1

        page = record["page"]

        # 곡 정보의 누락 위치 확인
        for field in ["song", "artist", "length"]:
            if field not in record:
                music_missing_by_page[page][field] += 1

                if page == "NextSong":
                    nextsong_missing[field] += 1

        # 가입 시각 누락의 발생 맥락 확인
        if "registration" not in record:
            auth = record["auth"]
            user_id = record["userId"]

            registration_missing_by_auth[auth] += 1
            registration_missing_by_user[user_id] += 1

        if event_count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at

            print(
                f"{event_count:,}건 확인 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

# 검사 결과 저장
missing_context_report = {
    "곡정보_누락_이벤트별": {
        page: dict(counts)
        for page, counts in music_missing_by_page.items()
    },
    "곡재생_이벤트의_누락": dict(nextsong_missing),
    "가입시각_누락_인증상태별": dict(
        registration_missing_by_auth
    ),
    "가입시각_누락_고유고객ID수": len(
        registration_missing_by_user
    ),
    "가입시각_누락_상위고객ID": (
        registration_missing_by_user.most_common(5)
    ),
}

save_path = REPORT_DIR / "02_누락_발생상황_검사.json"

with save_path.open("w", encoding="utf-8") as f:
    json.dump(
        missing_context_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n===== 누락 발생 상황 =====")

print("\n곡 재생 이벤트 안의 누락:")
for field in ["song", "artist", "length"]:
    print(f"{field}: {nextsong_missing[field]:,}건")

print("\n가입 시각이 없는 기록의 인증 상태:")
print(dict(registration_missing_by_auth))

print("\n가입 시각이 없는 기록의 고유 고객 ID 수:")
print(len(registration_missing_by_user))

print("\n가입 시각이 없는 기록의 상위 고객 ID 5개:")
for user_id, count in registration_missing_by_user.most_common(5):
    print(f"고객 ID {user_id!r}: {count:,}건")

print("\n보고서 저장:", save_path)
print("원본과 기존 CSV는 변경하지 않았습니다.")

5,000,000건 확인 중 | 0.9분
10,000,000건 확인 중 | 1.8분
15,000,000건 확인 중 | 2.6분
20,000,000건 확인 중 | 3.7분
25,000,000건 확인 중 | 4.9분

===== 누락 발생 상황 =====

곡 재생 이벤트 안의 누락:
song: 0건
artist: 0건
length: 0건

가입 시각이 없는 기록의 인증 상태:
{'Logged Out': 774285, 'Guest': 4194}

가입 시각이 없는 기록의 고유 고객 ID 수:
1

가입 시각이 없는 기록의 상위 고객 ID 5개:
고객 ID '1261737': 778,479건

보고서 저장: D:\Study\study\Project\2nd Project\reports\02_누락_발생상황_검사.json
원본과 기존 CSV는 변경하지 않았습니다.


In [8]:
import json
from collections import Counter

CHECK_USER_ID = "1261737"

# 인증 상태와 가입 정보 유무를 함께 집계
checked_counts = Counter()

# 가입 시각이 있는 경우 값이 일관적인지도 확인
registration_values = Counter()

checked_event_count = 0

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)

        if record["userId"] != CHECK_USER_ID:
            continue

        checked_event_count += 1

        auth = record["auth"]
        has_registration = "registration" in record

        checked_counts[(auth, has_registration)] += 1

        if has_registration:
            registration_values[record["registration"]] += 1

# 저장하기 쉬운 형태로 정리
check_rows = [
    {
        "인증 상태": auth,
        "가입 시각 항목 존재": has_registration,
        "이벤트 수": count,
    }
    for (auth, has_registration), count
    in checked_counts.items()
]

check_report = {
    "확인한 고객 ID": CHECK_USER_ID,
    "전체 이벤트 수": checked_event_count,
    "인증상태와_가입정보": check_rows,
    "서로 다른 가입 시각 값 수": len(registration_values),
}

save_path = REPORT_DIR / "03_비로그인_공통ID_점검.json"

with save_path.open("w", encoding="utf-8") as f:
    json.dump(
        check_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("===== 특정 ID 전체 기록 확인 =====")
print("확인한 ID:", CHECK_USER_ID)
print(f"전체 이벤트 수: {checked_event_count:,}")

for row in check_rows:
    print(
        f"인증 상태: {row['인증 상태']} | "
        f"가입 시각 항목 존재: {row['가입 시각 항목 존재']} | "
        f"이벤트 수: {row['이벤트 수']:,}"
    )

print("서로 다른 가입 시각 값 수:", len(registration_values))
print("보고서 저장:", save_path)
print("원본과 기존 CSV는 변경하지 않았습니다.")

===== 특정 ID 전체 기록 확인 =====
확인한 ID: 1261737
전체 이벤트 수: 778,479
인증 상태: Logged Out | 가입 시각 항목 존재: False | 이벤트 수: 774,285
인증 상태: Guest | 가입 시각 항목 존재: False | 이벤트 수: 4,194
서로 다른 가입 시각 값 수: 0
보고서 저장: D:\Study\study\Project\2nd Project\reports\03_비로그인_공통ID_점검.json
원본과 기존 CSV는 변경하지 않았습니다.


In [9]:
import json

# 원본 검사 결과에 근거한 제외 대상
EXCLUDED_USER_IDS = {"1261737"}

def keep_customer_event(record):
    """가입 고객 분석에서 사용할 기록이면 True를 반환합니다."""
    return record["userId"] not in EXCLUDED_USER_IDS


exclusion_policy = {
    "제외 고객 ID": sorted(EXCLUDED_USER_IDS),
    "확인된 제외 이벤트 수": 778_479,
    "제외 근거": [
        "모든 기록에 가입 시각 항목이 없음",
        "Logged Out 상태 774,285건",
        "Guest 상태 4,194건",
        "가입 고객의 행동으로 집계할 근거가 부족함",
    ],
    "적용 방법": (
        "이후 원본을 읽을 때 해당 ID의 기록을 "
        "피처 계산과 정답 생성에서 제외"
    ),
    "원본 수정 여부": False,
}

policy_path = REPORT_DIR / "04_고객분석_제외기준.json"

with policy_path.open("w", encoding="utf-8") as f:
    json.dump(
        exclusion_policy,
        f,
        ensure_ascii=False,
        indent=2
    )

# 제외 함수가 의도대로 작동하는지 확인
assert keep_customer_event({"userId": "1261737"}) is False
assert keep_customer_event({"userId": "확인용_다른_ID"}) is True

print("제외 기준 저장 완료")
print("제외 대상 ID:", sorted(EXCLUDED_USER_IDS))
print("확인된 제외 이벤트 수: 778,479건")
print("제외 함수 동작 확인 완료")
print("저장 위치:", policy_path)
print("아직 실제 데이터 파일은 변경하지 않았습니다.")

제외 기준 저장 완료
제외 대상 ID: ['1261737']
확인된 제외 이벤트 수: 778,479건
제외 함수 동작 확인 완료
저장 위치: D:\Study\study\Project\2nd Project\reports\04_고객분석_제외기준.json
아직 실제 데이터 파일은 변경하지 않았습니다.


In [10]:
import json
import time
from collections import Counter
from datetime import datetime, timezone

# 시간 단위 오류를 찾기 위한 넓은 점검 범위
# 범위를 벗어나도 자동 삭제하지 않습니다.
time_lower_ms = int(
    datetime(2000, 1, 1, tzinfo=timezone.utc).timestamp() * 1000
)
time_upper_ms = int(
    datetime(2100, 1, 1, tzinfo=timezone.utc).timestamp() * 1000
)

numeric_fields = [
    "ts",
    "registration",
    "sessionId",
    "itemInSession",
    "status",
    "length",
]

numeric_ranges = {
    field: {"최솟값": None, "최댓값": None}
    for field in numeric_fields
}

range_issues = Counter({
    "음수 세션 번호": 0,
    "음수 세션 내 순번": 0,
    "0 이하 곡 길이": 0,
    "이벤트보다 늦은 가입 시각": 0,
    "이벤트 시각 점검 범위 이탈": 0,
    "가입 시각 점검 범위 이탈": 0,
    "웹 응답 코드 범위 이탈": 0,
})

category_counts = {
    field: Counter()
    for field in ["level", "auth", "method", "page"]
}

read_count = 0
included_count = 0
excluded_count = 0

started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        read_count += 1

        if read_count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"{read_count:,}건 확인 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

        # 앞 단계에서 확정한 제외 기준 적용
        if not keep_customer_event(record):
            excluded_count += 1
            continue

        included_count += 1

        # 숫자 항목의 최솟값·최댓값
        for field in numeric_fields:
            value = record.get(field)

            # 곡 재생이 아닌 이벤트에는 length가 없을 수 있습니다.
            if value is None:
                continue

            current = numeric_ranges[field]

            if current["최솟값"] is None or value < current["최솟값"]:
                current["최솟값"] = value

            if current["최댓값"] is None or value > current["최댓값"]:
                current["최댓값"] = value

        # 기본 범위 검사
        if record["sessionId"] < 0:
            range_issues["음수 세션 번호"] += 1

        if record["itemInSession"] < 0:
            range_issues["음수 세션 내 순번"] += 1

        length = record.get("length")
        if length is not None and length <= 0:
            range_issues["0 이하 곡 길이"] += 1

        ts = record["ts"]
        registration = record.get("registration")

        if not time_lower_ms <= ts < time_upper_ms:
            range_issues["이벤트 시각 점검 범위 이탈"] += 1

        if registration is not None:
            if registration > ts:
                range_issues["이벤트보다 늦은 가입 시각"] += 1

            if not time_lower_ms <= registration < time_upper_ms:
                range_issues["가입 시각 점검 범위 이탈"] += 1

        if not 100 <= record["status"] <= 599:
            range_issues["웹 응답 코드 범위 이탈"] += 1

        # 범주 값은 아직 공백 제거 등으로 수정하지 않습니다.
        for field in category_counts:
            category_counts[field][record[field]] += 1

# 기록 수 일치 확인
assert read_count == included_count + excluded_count
assert excluded_count == 778_479

value_report = {
    "읽은 이벤트 수": read_count,
    "제외 이벤트 수": excluded_count,
    "검사 대상 이벤트 수": included_count,
    "숫자 범위": numeric_ranges,
    "범위 확인 사항": dict(range_issues),
    "범주별 건수": {
        field: dict(counts)
        for field, counts in category_counts.items()
    },
}

save_path = REPORT_DIR / "05_숫자범위_범주값_검사.json"

with save_path.open("w", encoding="utf-8") as f:
    json.dump(
        value_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n===== 숫자 범위·범주 값 확인 =====")
print(f"검사 대상: {included_count:,}건")
print(f"제외 기록: {excluded_count:,}건")

print("\n숫자 범위:")
for field, values in numeric_ranges.items():
    print(
        f"{field}: "
        f"{values['최솟값']} ~ {values['최댓값']}"
    )

print("\n범위 확인 사항:")
print(dict(range_issues))

print("\n실제 범주 값:")
for field, counts in category_counts.items():
    print(f"{field}: {list(counts.keys())}")

print("\n보고서 저장:", save_path)
print("원본과 기존 CSV는 변경하지 않았습니다.")

5,000,000건 확인 중 | 2.0분
10,000,000건 확인 중 | 4.1분
15,000,000건 확인 중 | 6.3분
20,000,000건 확인 중 | 8.4분
25,000,000건 확인 중 | 10.6분

===== 숫자 범위·범주 값 확인 =====
검사 대상: 25,480,720건
제외 기록: 778,479건

숫자 범위:
ts: 1538352001000 ~ 1543622402000
registration: 1508018725000 ~ 1543821822000
sessionId: 1 ~ 240381
itemInSession: 0 ~ 1426
status: 200 ~ 404
length: 0.522 ~ 3024.66567

범위 확인 사항:
{'음수 세션 번호': 0, '음수 세션 내 순번': 0, '0 이하 곡 길이': 0, '이벤트보다 늦은 가입 시각': 11484, '이벤트 시각 점검 범위 이탈': 0, '가입 시각 점검 범위 이탈': 0, '웹 응답 코드 범위 이탈': 0}

실제 범주 값:
level: ['paid', 'free']
auth: ['Logged In', 'Cancelled']
method: ['PUT', 'GET']
page: ['NextSong', 'Home', 'Thumbs Up', 'Error', 'Help', 'Add to Playlist', 'Roll Advert', 'Thumbs Down', 'Settings', 'About', 'Add Friend', 'Upgrade', 'Submit Upgrade', 'Logout', 'Downgrade', 'Save Settings', 'Submit Downgrade', 'Cancel', 'Cancellation Confirmation']

보고서 저장: D:\Study\study\Project\2nd Project\reports\05_숫자범위_범주값_검사.json
원본과 기존 CSV는 변경하지 않았습니다.


In [11]:
import json
import time

# 고객별 시간 정보만 모읍니다.
# 원본 이벤트 전체를 메모리에 저장하지 않습니다.
customer_time_checks = {}

read_count = 0
started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        read_count += 1

        if read_count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"{read_count:,}건 확인 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

        if not keep_customer_event(record):
            continue

        user_id = record["userId"]
        ts = record["ts"]
        registration = record.get("registration")

        if user_id not in customer_time_checks:
            customer_time_checks[user_id] = {
                "event_count": 0,
                "late_registration_events": 0,
                "registration_values": set(),
                "first_event_ts": ts,
                "last_event_ts": ts,
                "max_late_gap_ms": 0,
            }

        stats = customer_time_checks[user_id]

        stats["event_count"] += 1
        stats["first_event_ts"] = min(
            stats["first_event_ts"], ts
        )
        stats["last_event_ts"] = max(
            stats["last_event_ts"], ts
        )

        if registration is not None:
            stats["registration_values"].add(registration)

            if registration > ts:
                stats["late_registration_events"] += 1

                stats["max_late_gap_ms"] = max(
                    stats["max_late_gap_ms"],
                    registration - ts
                )

# 문제가 확인된 고객만 별도로 정리
affected_rows = []

for user_id, stats in customer_time_checks.items():
    if stats["late_registration_events"] == 0:
        continue

    affected_rows.append({
        "user_id": user_id,
        "전체 이벤트 수": stats["event_count"],
        "가입 이전 이벤트 수": stats["late_registration_events"],
        "서로 다른 가입 시각 수": len(
            stats["registration_values"]
        ),
        "최대 시간 차이_일": round(
            stats["max_late_gap_ms"] / 86_400_000, 4
        ),
    })

affected_event_count = sum(
    row["가입 이전 이벤트 수"]
    for row in affected_rows
)

# 앞 단계와 같은 건수인지 확인
assert affected_event_count == 11_484

all_events_before_registration = sum(
    row["가입 이전 이벤트 수"] == row["전체 이벤트 수"]
    for row in affected_rows
)

multiple_registration_users = sum(
    row["서로 다른 가입 시각 수"] > 1
    for row in affected_rows
)

# 영향받은 고객뿐 아니라 전체 고객의 가입 시각 일관성도 확인
all_multiple_registration_users = sum(
    len(stats["registration_values"]) > 1
    for stats in customer_time_checks.values()
)

registration_report = {
    "검사한 고객 수": len(customer_time_checks),
    "가입 이전 이벤트 수": affected_event_count,
    "문제가 확인된 고객 수": len(affected_rows),
    "모든 이벤트가 가입 시각보다 앞선 고객 수": (
        all_events_before_registration
    ),
    "문제 고객 중 가입 시각이 여러 개인 고객 수": (
        multiple_registration_users
    ),
    "전체 고객 중 가입 시각이 여러 개인 고객 수": (
        all_multiple_registration_users
    ),
    "문제 고객 상세": affected_rows,
}

save_path = REPORT_DIR / "06_가입시각_정합성_검사.json"

with save_path.open("w", encoding="utf-8") as f:
    json.dump(
        registration_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n===== 가입 시각 문제 확인 =====")
print(f"검사한 고객 수: {len(customer_time_checks):,}")
print(f"가입 이전 이벤트 수: {affected_event_count:,}")
print(f"문제가 확인된 고객 수: {len(affected_rows):,}")

print(
    "모든 이벤트가 가입 시각보다 앞선 고객 수:",
    all_events_before_registration
)
print(
    "문제 고객 중 가입 시각이 여러 개인 고객 수:",
    multiple_registration_users
)
print(
    "전체 고객 중 가입 시각이 여러 개인 고객 수:",
    all_multiple_registration_users
)

print("\n최대 시간 차이가 큰 사례 5개:")
for row in sorted(
    affected_rows,
    key=lambda row: row["최대 시간 차이_일"],
    reverse=True
)[:5]:
    print(row)

print("\n보고서 저장:", save_path)
print("원본과 기존 CSV는 변경하지 않았습니다.")

5,000,000건 확인 중 | 1.2분
10,000,000건 확인 중 | 2.4분
15,000,000건 확인 중 | 4.1분
20,000,000건 확인 중 | 5.8분
25,000,000건 확인 중 | 6.9분

===== 가입 시각 문제 확인 =====
검사한 고객 수: 22,277
가입 이전 이벤트 수: 11,484
문제가 확인된 고객 수: 144
모든 이벤트가 가입 시각보다 앞선 고객 수: 19
문제 고객 중 가입 시각이 여러 개인 고객 수: 0
전체 고객 중 가입 시각이 여러 개인 고객 수: 0

최대 시간 차이가 큰 사례 5개:
{'user_id': '1516373', '전체 이벤트 수': 357, '가입 이전 이벤트 수': 357, '서로 다른 가입 시각 수': 1, '최대 시간 차이_일': 22.7955}
{'user_id': '1835558', '전체 이벤트 수': 144, '가입 이전 이벤트 수': 144, '서로 다른 가입 시각 수': 1, '최대 시간 차이_일': 19.7256}
{'user_id': '1039566', '전체 이벤트 수': 154, '가입 이전 이벤트 수': 91, '서로 다른 가입 시각 수': 1, '최대 시간 차이_일': 18.4004}
{'user_id': '1209694', '전체 이벤트 수': 66, '가입 이전 이벤트 수': 20, '서로 다른 가입 시각 수': 1, '최대 시간 차이_일': 15.6545}
{'user_id': '1075367', '전체 이벤트 수': 15, '가입 이전 이벤트 수': 15, '서로 다른 가입 시각 수': 1, '최대 시간 차이_일': 14.2241}

보고서 저장: D:\Study\study\Project\2nd Project\reports\06_가입시각_정합성_검사.json
원본과 기존 CSV는 변경하지 않았습니다.


In [12]:
import json

# 가입 시각을 사용하는 피처는 이번 버전에서 만들지 않습니다.
USE_REGISTRATION_FEATURES = False

registration_policy = {
    "문제가 확인된 고객 수": 144,
    "가입 이전 이벤트 수": 11_484,
    "모든 이벤트가 가입 시각보다 앞선 고객 수": 19,
    "확인 결과": (
        "모든 고객의 가입 시각은 고객별로 한 가지였으나, "
        "일부 고객에서 이벤트 시각과 모순됨"
    ),
    "처리 기준": {
        "가입 시각 기반 피처 사용": False,
        "가입 시각 임의 수정": False,
        "이 문제만을 근거로 고객 전체 삭제": False,
        "행동 및 해지 기록": "보존하되 시간 정보의 한계를 보고서에 명시",
    },
    "제외할 피처 예시": [
        "가입 후 경과 일수",
        "가입 기간 대비 활동량",
    ],
    "판단의 한계": (
        "가입 시각과 이벤트 시각 중 어느 값이 잘못됐는지 "
        "현재 자료만으로 확정할 수 없음"
    ),
}

policy_path = REPORT_DIR / "07_가입시각_처리기준.json"

with policy_path.open("w", encoding="utf-8") as f:
    json.dump(
        registration_policy,
        f,
        ensure_ascii=False,
        indent=2
    )

print("가입 시각 처리 기준 저장 완료")
print("가입 시각 기반 피처 사용:", USE_REGISTRATION_FEATURES)
print("해당 고객의 행동·해지 기록은 삭제하지 않습니다.")
print("가입 시각을 임의로 수정하지 않습니다.")
print("저장 위치:", policy_path)
print("원본과 기존 CSV는 변경하지 않았습니다.")

가입 시각 처리 기준 저장 완료
가입 시각 기반 피처 사용: False
해당 고객의 행동·해지 기록은 삭제하지 않습니다.
가입 시각을 임의로 수정하지 않습니다.
저장 위치: D:\Study\study\Project\2nd Project\reports\07_가입시각_처리기준.json
원본과 기존 CSV는 변경하지 않았습니다.


In [13]:
import json
from datetime import datetime, timezone

# 관찰 시작
OBS_START = datetime(
    2018, 10, 4, tzinfo=timezone.utc
)

# 예측 기준일
CUTOFF = datetime(
    2018, 11, 1, tzinfo=timezone.utc
)

# 예측 기간 종료
PRED_END = datetime(
    2018, 11, 15, tzinfo=timezone.utc
)

# 원본 ts와 비교하기 위한 밀리초 값
OBS_START_MS = int(OBS_START.timestamp() * 1000)
CUTOFF_MS = int(CUTOFF.timestamp() * 1000)
PRED_END_MS = int(PRED_END.timestamp() * 1000)

# 서비스 해지 완료를 나타내는 이벤트
CHURN_EVENT = "Cancellation Confirmation"

# 앞서 저장한 검사 결과로 원본 날짜 범위 확인
range_report_path = (
    REPORT_DIR / "05_숫자범위_범주값_검사.json"
)

with range_report_path.open("r", encoding="utf-8") as f:
    range_report = json.load(f)

source_start_ms = range_report["숫자 범위"]["ts"]["최솟값"]
source_end_ms = range_report["숫자 범위"]["ts"]["최댓값"]

assert OBS_START_MS < CUTOFF_MS < PRED_END_MS
assert source_start_ms <= OBS_START_MS
assert PRED_END_MS <= source_end_ms

analysis_config = {
    "시간대": "UTC",
    "관찰 시작": OBS_START.isoformat(),
    "예측 기준일": CUTOFF.isoformat(),
    "예측 종료": PRED_END.isoformat(),
    "관찰 기간_일": (CUTOFF - OBS_START).days,
    "예측 기간_일": (PRED_END - CUTOFF).days,
    "기간 경계": "시작 포함, 끝 제외",
    "해지 완료 이벤트": CHURN_EVENT,
    "가입 시각 기반 피처 사용": USE_REGISTRATION_FEATURES,
    "제외 고객 ID": sorted(EXCLUDED_USER_IDS),
}

config_path = REPORT_DIR / "08_분석기간_설정.json"

with config_path.open("w", encoding="utf-8") as f:
    json.dump(
        analysis_config,
        f,
        ensure_ascii=False,
        indent=2
    )

print("분석 기간 설정 완료")
print("관찰 시작:", OBS_START.isoformat())
print("예측 기준일:", CUTOFF.isoformat())
print("예측 종료:", PRED_END.isoformat())
print("관찰 기간:", (CUTOFF - OBS_START).days, "일")
print("예측 기간:", (PRED_END - CUTOFF).days, "일")
print("원본 날짜 범위 안에 있는지 확인 완료")
print("설정 저장:", config_path)

분석 기간 설정 완료
관찰 시작: 2018-10-04T00:00:00+00:00
예측 기준일: 2018-11-01T00:00:00+00:00
예측 종료: 2018-11-15T00:00:00+00:00
관찰 기간: 28 일
예측 기간: 14 일
원본 날짜 범위 안에 있는지 확인 완료
설정 저장: D:\Study\study\Project\2nd Project\reports\08_분석기간_설정.json


In [15]:
import json
import time

# 관찰 기간에 활동한 고객
observed_users = set()

# 기준일 전에 해지한 고객
previously_churned_users = set()

# 예측 기간에 해지한 고객
prediction_period_churned_users = set()

read_count = 0
excluded_event_count = 0

started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line in f:
        if not line.strip():
            continue

        record = json.loads(line)
        read_count += 1

        if read_count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"{read_count:,}건 확인 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

        if not keep_customer_event(record):
            excluded_event_count += 1
            continue

        user_id = record["userId"]
        ts = record["ts"]
        page = record["page"]

        # 관찰 기간에 기록이 있는 고객
        if OBS_START_MS <= ts < CUTOFF_MS:
            observed_users.add(user_id)

        # 해지 완료 사건만 정답 구성에 사용
        if page == CHURN_EVENT:
            if ts < CUTOFF_MS:
                previously_churned_users.add(user_id)

            elif CUTOFF_MS <= ts < PRED_END_MS:
                prediction_period_churned_users.add(user_id)

# 관찰 고객 중 기준일 이전 해지 고객 제외
eligible_users = observed_users - previously_churned_users

# 분석 대상 안에서 정답 구성
positive_users = (
    eligible_users & prediction_period_churned_users
)
negative_users = eligible_users - positive_users

assert eligible_users
assert eligible_users.isdisjoint(EXCLUDED_USER_IDS)
assert eligible_users.isdisjoint(previously_churned_users)
assert positive_users.isdisjoint(negative_users)
assert positive_users | negative_users == eligible_users

# 고객 목록과 정답 저장
cohort_rows = [
    {
        "user_id": user_id,
        "churn": int(user_id in positive_users),
    }
    for user_id in sorted(eligible_users)
]

cohort_path = PROCESSED_DIR / "customer_cohort.json"

with cohort_path.open("w", encoding="utf-8") as f:
    json.dump(
        cohort_rows,
        f,
        ensure_ascii=False,
        indent=2
    )

cohort_report = {
    "관찰 기간 활동 고객 수": len(observed_users),
    "관찰 고객 중 기준일 이전 해지로 제외한 고객 수": len(
        observed_users & previously_churned_users
    ),
    "최종 분석 대상 고객 수": len(eligible_users),
    "이탈 정답 1 고객 수": len(positive_users),
    "이탈 정답 0 고객 수": len(negative_users),
    "이탈 비율": len(positive_users) / len(eligible_users),
    "제외 ID의 이벤트 수": excluded_event_count,
}

report_path = REPORT_DIR / "09_분석대상_정답라벨.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        cohort_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n===== 분석 대상과 이탈 정답 확정 =====")
print(f"관찰 기간 활동 고객: {len(observed_users):,}명")

print(
    "기준일 이전 해지로 제외:",
    f"{len(observed_users & previously_churned_users):,}명"
)

print(f"최종 분석 대상: {len(eligible_users):,}명")
print(f"이탈 정답 1: {len(positive_users):,}명")
print(f"이탈 정답 0: {len(negative_users):,}명")

print(
    "이탈 비율:",
    f"{len(positive_users) / len(eligible_users):.2%}"
)

print("고객 목록 저장:", cohort_path)
print("보고서 저장:", report_path)

5,000,000건 확인 중 | 1.0분
10,000,000건 확인 중 | 2.5분
15,000,000건 확인 중 | 4.0분
20,000,000건 확인 중 | 5.4분
25,000,000건 확인 중 | 6.8분

===== 분석 대상과 이탈 정답 확정 =====
관찰 기간 활동 고객: 20,642명
기준일 이전 해지로 제외: 2,503명
최종 분석 대상: 18,139명
이탈 정답 1: 1,035명
이탈 정답 0: 17,104명
이탈 비율: 5.71%
고객 목록 저장: D:\Study\study\Project\2nd Project\data\processed\customer_cohort.json
보고서 저장: D:\Study\study\Project\2nd Project\reports\09_분석대상_정답라벨.json


In [16]:
import json

FEATURE_DEFINITIONS = {
    "active_days": {
        "종류": "수치형",
        "설명": "관찰 기간에 활동한 서로 다른 UTC 날짜 수",
    },
    "session_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 고객별 서로 다른 세션 수",
    },
    "song_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 NextSong 이벤트 수",
    },
    "days_since_last_activity": {
        "종류": "수치형",
        "설명": "마지막 활동부터 예측 기준일까지 경과한 일수",
    },
    "recent_7d_song_count": {
        "종류": "수치형",
        "설명": "예측 기준일 직전 7일의 NextSong 이벤트 수",
    },
    "previous_7d_song_count": {
        "종류": "수치형",
        "설명": "최근 7일보다 앞선 7일의 NextSong 이벤트 수",
    },
    "song_count_change": {
        "종류": "수치형",
        "설명": "최근 7일 재생 횟수에서 이전 7일 재생 횟수를 뺀 값",
    },
    "thumbs_up_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Thumbs Up 이벤트 수",
    },
    "thumbs_down_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Thumbs Down 이벤트 수",
    },
    "playlist_add_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Add to Playlist 이벤트 수",
    },
    "ad_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Roll Advert 이벤트 수",
    },
    "error_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Error 이벤트 수",
    },
    "help_count": {
        "종류": "수치형",
        "설명": "관찰 기간의 Help 이벤트 수",
    },
    "level_last": {
        "종류": "범주형",
        "설명": "予測基準日前の観察期間内で最後に確認した無料・有料状態",
    },
}

# 마지막 이용 등급의 설명
FEATURE_DEFINITIONS["level_last"]["설명"] = (
    "예측 기준일 이전 관찰 기간에서 마지막으로 확인한 "
    "무료·유료 상태"
)

feature_spec = {
    "공통 기준": [
        "모든 피처는 관찰 기간의 기록만 사용",
        "이벤트 시각과 날짜는 UTC 기준",
        "가입 시각 기반 피처는 제외",
        "고객 ID와 이탈 정답은 입력 피처에서 제외",
        "곡 재생 이벤트 수를 완청 횟수나 실제 청취 시간으로 해석하지 않음",
    ],
    "피처 정의": FEATURE_DEFINITIONS,
}

spec_path = REPORT_DIR / "10_피처_정의서.json"

with spec_path.open("w", encoding="utf-8") as f:
    json.dump(
        feature_spec,
        f,
        ensure_ascii=False,
        indent=2
    )

numeric_count = sum(
    item["종류"] == "수치형"
    for item in FEATURE_DEFINITIONS.values()
)

categorical_count = sum(
    item["종류"] == "범주형"
    for item in FEATURE_DEFINITIONS.values()
)

print("피처 정의서 저장 완료")
print("수치형 피처:", numeric_count, "개")
print("범주형 피처:", categorical_count, "개")
print("전체 입력 피처:", len(FEATURE_DEFINITIONS), "개")
print("저장 위치:", spec_path)

피처 정의서 저장 완료
수치형 피처: 13 개
범주형 피처: 1 개
전체 입력 피처: 14 개
저장 위치: D:\Study\study\Project\2nd Project\reports\10_피처_정의서.json


In [17]:
import json
import time
import heapq
from collections import defaultdict

DAY_MS = 86_400_000
MAX_SEQUENCE_LENGTH = 50

RECENT_7D_START_MS = CUTOFF_MS - 7 * DAY_MS
PREVIOUS_7D_START_MS = CUTOFF_MS - 14 * DAY_MS

# 이벤트 이름과 계산할 피처 이름의 연결
EVENT_FEATURE_MAP = {
    "NextSong": "song_count",
    "Thumbs Up": "thumbs_up_count",
    "Thumbs Down": "thumbs_down_count",
    "Add to Playlist": "playlist_add_count",
    "Roll Advert": "ad_count",
    "Error": "error_count",
    "Help": "help_count",
}


def new_customer_stats():
    stats = {
        "active_dates": set(),
        "session_ids": set(),
        "last_event_key": None,
        "level_last": None,
        "recent_7d_song_count": 0,
        "previous_7d_song_count": 0,
        "recent_events": [],
    }

    for feature_name in EVENT_FEATURE_MAP.values():
        stats[feature_name] = 0

    return stats


customer_stats = defaultdict(new_customer_stats)

read_count = 0
used_event_count = 0
started_at = time.perf_counter()

with RAW_PATH.open("r", encoding="utf-8-sig") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        record = json.loads(line)
        read_count += 1

        if read_count % 5_000_000 == 0:
            elapsed = time.perf_counter() - started_at
            print(
                f"{read_count:,}건 확인 중 "
                f"| {elapsed / 60:.1f}분",
                flush=True
            )

        if not keep_customer_event(record):
            continue

        user_id = record["userId"]
        ts = record["ts"]

        # 확정된 고객과 관찰 기간만 사용
        if user_id not in eligible_users:
            continue

        if not OBS_START_MS <= ts < CUTOFF_MS:
            continue

        page = record["page"]

        # 기준일 이전 해지 고객이 섞이지 않았는지 확인
        if page == CHURN_EVENT or record["auth"] == "Cancelled":
            raise ValueError(
                "관찰 기간의 분석 대상에 해지 상태가 있습니다. "
                "고객 선정 기준을 다시 확인해야 합니다."
            )

        stats = customer_stats[user_id]
        used_event_count += 1

        # 1. UTC 기준 활동 날짜
        stats["active_dates"].add(ts // DAY_MS)

        # 2. 고객별 서로 다른 세션
        stats["session_ids"].add(record["sessionId"])

        # 3. 이벤트 종류별 횟수
        feature_name = EVENT_FEATURE_MAP.get(page)

        if feature_name is not None:
            stats[feature_name] += 1

        # 4. 최근 두 주의 곡 재생 횟수
        if page == "NextSong":
            if RECENT_7D_START_MS <= ts < CUTOFF_MS:
                stats["recent_7d_song_count"] += 1

            elif PREVIOUS_7D_START_MS <= ts < RECENT_7D_START_MS:
                stats["previous_7d_song_count"] += 1

        # 같은 시각의 기록은 원본 줄 번호로 순서를 구분
        event_key = (ts, line_number)

        # 5. 마지막 이벤트 시각과 당시 이용 등급
        if (
            stats["last_event_key"] is None
            or event_key > stats["last_event_key"]
        ):
            stats["last_event_key"] = event_key
            stats["level_last"] = record["level"]

        # 6. 최근 행동 최대 50개 보관
        # 가장 오래된 기록부터 교체하는 구조
        sequence_item = (ts, line_number, page)
        recent_events = stats["recent_events"]

        if len(recent_events) < MAX_SEQUENCE_LENGTH:
            heapq.heappush(recent_events, sequence_item)

        elif sequence_item > recent_events[0]:
            heapq.heapreplace(recent_events, sequence_item)

# 분석 대상 모두에게 집계 결과가 있는지 확인
assert set(customer_stats) == eligible_users

# 보관한 최근 행동을 실제 시간 순서로 정렬
for stats in customer_stats.values():
    stats["recent_events"].sort()

    assert 1 <= len(stats["active_dates"]) <= 28
    assert len(stats["session_ids"]) >= 1
    assert stats["level_last"] in {"free", "paid"}

    assert (
        stats["recent_7d_song_count"]
        + stats["previous_7d_song_count"]
        <= stats["song_count"]
    )

    assert 1 <= len(stats["recent_events"]) <= 50

    assert all(
        OBS_START_MS <= event[0] < CUTOFF_MS
        for event in stats["recent_events"]
    )

short_sequence_count = sum(
    len(stats["recent_events"]) < MAX_SEQUENCE_LENGTH
    for stats in customer_stats.values()
)

elapsed = time.perf_counter() - started_at

print("\n===== 고객별 행동 집계 완료 =====")
print(f"집계 고객 수: {len(customer_stats):,}명")
print(f"사용한 이벤트 수: {used_event_count:,}건")
print(f"행동 기록이 50개보다 적은 고객: {short_sequence_count:,}명")
print("고객 누락·관찰 기간·기본 집계 검사 통과")
print(f"실행 시간: {elapsed / 60:.1f}분")
print("아직 원본과 기존 CSV는 변경하지 않았습니다.")

5,000,000건 확인 중 | 1.0분
10,000,000건 확인 중 | 2.2분
15,000,000건 확인 중 | 2.9분
20,000,000건 확인 중 | 3.6분
25,000,000건 확인 중 | 4.4분

===== 고객별 행동 집계 완료 =====
집계 고객 수: 18,139명
사용한 이벤트 수: 10,781,540건
행동 기록이 50개보다 적은 고객: 1,859명
고객 누락·관찰 기간·기본 집계 검사 통과
실행 시간: 4.6분
아직 원본과 기존 CSV는 변경하지 않았습니다.


In [18]:
import math
from collections import Counter

customer_rows = []

for user_id in sorted(eligible_users):
    stats = customer_stats[user_id]

    last_activity_ts = stats["last_event_key"][0]

    row = {
        "user_id": user_id,

        "active_days": len(stats["active_dates"]),
        "session_count": len(stats["session_ids"]),
        "song_count": stats["song_count"],

        "days_since_last_activity": round(
            (CUTOFF_MS - last_activity_ts) / DAY_MS,
            6
        ),

        "recent_7d_song_count": stats["recent_7d_song_count"],
        "previous_7d_song_count": stats["previous_7d_song_count"],

        "song_count_change": (
            stats["recent_7d_song_count"]
            - stats["previous_7d_song_count"]
        ),

        "thumbs_up_count": stats["thumbs_up_count"],
        "thumbs_down_count": stats["thumbs_down_count"],
        "playlist_add_count": stats["playlist_add_count"],
        "ad_count": stats["ad_count"],
        "error_count": stats["error_count"],
        "help_count": stats["help_count"],

        "level_last": stats["level_last"],

        "churn": int(user_id in positive_users),
    }

    customer_rows.append(row)


# ---------- 검증 ----------

expected_columns = (
    {"user_id", "churn"} | set(FEATURE_DEFINITIONS)
)

numeric_features = [
    name
    for name, definition in FEATURE_DEFINITIONS.items()
    if definition["종류"] == "수치형"
]

# 음수가 되면 안 되는 횟수 항목
count_features = [
    "active_days",
    "session_count",
    "song_count",
    "recent_7d_song_count",
    "previous_7d_song_count",
    "thumbs_up_count",
    "thumbs_down_count",
    "playlist_add_count",
    "ad_count",
    "error_count",
    "help_count",
]

# 고객 수·중복·누락 확인
assert len(customer_rows) == len(eligible_users) == 18_139
assert len({row["user_id"] for row in customer_rows}) == 18_139

assert {
    row["user_id"] for row in customer_rows
} == eligible_users

for row in customer_rows:

    # 열 구성이 피처 정의서와 일치하는지 확인
    assert set(row) == expected_columns

    # 고객 ID 확인
    assert isinstance(row["user_id"], str)
    assert row["user_id"].strip() != ""
    assert row["user_id"] == row["user_id"].strip()

    # 수치형 피처의 자료형·결측·무한대 확인
    for feature in numeric_features:
        value = row[feature]

        assert isinstance(value, (int, float))
        assert not isinstance(value, bool)
        assert math.isfinite(value)

    # 횟수 피처는 음수가 아닌 정수
    for feature in count_features:
        assert isinstance(row[feature], int)
        assert row[feature] >= 0

    assert 1 <= row["active_days"] <= 28
    assert row["session_count"] >= 1

    assert 0 <= row["days_since_last_activity"] <= 28

    # 두 주의 재생 횟수 합은 전체 기간 횟수 이하
    assert (
        row["recent_7d_song_count"]
        + row["previous_7d_song_count"]
        <= row["song_count"]
    )

    # 변화량은 음수일 수 있지만 정수여야 함
    assert isinstance(row["song_count_change"], int)

    assert row["song_count_change"] == (
        row["recent_7d_song_count"]
        - row["previous_7d_song_count"]
    )

    # 범주형과 정답 확인
    assert row["level_last"] in {"free", "paid"}
    assert row["churn"] in (0, 1)

assert sum(row["churn"] for row in customer_rows) == 1_035

level_counts = Counter(
    row["level_last"] for row in customer_rows
)

print("===== 고객별 표 생성 및 검증 완료 =====")
print(f"고객 수: {len(customer_rows):,}명")
print(f"열 수: {len(customer_rows[0])}개")
print(f"입력 피처 수: {len(FEATURE_DEFINITIONS)}개")
print(f"이탈 고객 수: {sum(r['churn'] for r in customer_rows):,}명")
print("마지막 이용 등급별 고객 수:", dict(level_counts))
print("자료형·빈값·무한대·기본 범위·정답 검사 통과")

print("\n첫 번째 고객 예시:")
print(json.dumps(
    customer_rows[0],
    ensure_ascii=False,
    indent=2
))

print("\n아직 CSV 파일은 저장하지 않았습니다.")

===== 고객별 표 생성 및 검증 완료 =====
고객 수: 18,139명
열 수: 16개
입력 피처 수: 14개
이탈 고객 수: 1,035명
마지막 이용 등급별 고객 수: {'paid': 9678, 'free': 8461}
자료형·빈값·무한대·기본 범위·정답 검사 통과

첫 번째 고객 예시:
{
  "user_id": "1000035",
  "active_days": 14,
  "session_count": 14,
  "song_count": 692,
  "days_since_last_activity": 3.244144,
  "recent_7d_song_count": 221,
  "previous_7d_song_count": 187,
  "song_count_change": 34,
  "thumbs_up_count": 57,
  "thumbs_down_count": 9,
  "playlist_add_count": 15,
  "ad_count": 5,
  "error_count": 0,
  "help_count": 3,
  "level_last": "paid",
  "churn": 0
}

아직 CSV 파일은 저장하지 않았습니다.


In [19]:
import csv
import json
import math

feature_columns = list(FEATURE_DEFINITIONS)

columns = ["user_id"] + feature_columns + ["churn"]

save_path = (
    PROCESSED_DIR / "customer_features_20181101_v1.csv"
)

# 1. CSV 저장
with save_path.open(
    "w", encoding="utf-8-sig", newline=""
) as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(customer_rows)

# 2. 저장된 CSV를 다시 읽기
with save_path.open(
    "r", encoding="utf-8-sig", newline=""
) as f:
    reader = csv.DictReader(f)
    saved_columns = reader.fieldnames
    saved_rows = list(reader)

# 3. 열 구성과 고객 수 확인
assert saved_columns == columns
assert len(saved_rows) == len(customer_rows) == 18_139

saved_user_ids = [row["user_id"] for row in saved_rows]

assert len(set(saved_user_ids)) == len(saved_user_ids)
assert set(saved_user_ids) == eligible_users

# 4. 각 값이 저장 전과 일치하는지 확인
# CSV는 다시 읽으면 값이 문자열이므로,
# 숫자 항목은 숫자로 해석한 뒤 비교합니다.
for original, saved in zip(customer_rows, saved_rows):
    assert saved["user_id"] == original["user_id"]
    assert int(saved["churn"]) == original["churn"]

    for feature in feature_columns:
        original_value = original[feature]
        saved_value = saved[feature]

        if FEATURE_DEFINITIONS[feature]["종류"] == "범주형":
            assert saved_value == original_value

        elif isinstance(original_value, int):
            assert int(saved_value) == original_value

        else:
            restored_value = float(saved_value)

            assert math.isfinite(restored_value)
            assert restored_value == original_value

assert sum(int(row["churn"]) for row in saved_rows) == 1_035

# 5. 저장 확인 결과 기록
save_report = {
    "파일": str(save_path),
    "고객 수": len(saved_rows),
    "열 수": len(saved_columns),
    "입력 피처 수": len(feature_columns),
    "이탈 고객 수": 1_035,
    "저장 전후 값 일치": True,
    "학습_검증_최종평가_분할": "아직 적용하지 않음",
}

report_path = REPORT_DIR / "11_고객별CSV_저장확인.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        save_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("===== CSV 저장 및 재검사 완료 =====")
print("저장 위치:", save_path)
print(f"고객 수: {len(saved_rows):,}명")
print(f"열 수: {len(saved_columns)}개")
print("저장 전후 고객·피처·정답 일치 확인")
print("저장 확인 보고서:", report_path)

===== CSV 저장 및 재검사 완료 =====
저장 위치: D:\Study\study\Project\2nd Project\data\processed\customer_features_20181101_v1.csv
고객 수: 18,139명
열 수: 16개
저장 전후 고객·피처·정답 일치 확인
저장 확인 보고서: D:\Study\study\Project\2nd Project\reports\11_고객별CSV_저장확인.json


In [20]:
import pandas as pd
from sklearn.model_selection import train_test_split

feature_path = (
    PROCESSED_DIR / "customer_features_20181101_v1.csv"
)

df = pd.read_csv(
    feature_path,
    dtype={
        "user_id": "string",
        "level_last": "string",
    }
)

assert len(df) == 18_139
assert df["user_id"].is_unique
assert df["churn"].isin([0, 1]).all()
assert df["churn"].sum() == 1_035

# 1. 학습 70%, 나머지 30%로 분리
train_idx, remaining_idx = train_test_split(
    df.index.to_numpy(),
    test_size=0.30,
    random_state=42,
    stratify=df["churn"].to_numpy(),
)

# 2. 나머지를 검증 15%, 최종 평가 15%로 분리
valid_idx, test_idx = train_test_split(
    remaining_idx,
    test_size=0.50,
    random_state=42,
    stratify=df.loc[remaining_idx, "churn"].to_numpy(),
)

# 3. 고객별 분할 표시
df["split"] = ""

df.loc[train_idx, "split"] = "train"
df.loc[valid_idx, "split"] = "valid"
df.loc[test_idx, "split"] = "test"

# 4. 누락·중복 확인
train_users = set(df.loc[train_idx, "user_id"])
valid_users = set(df.loc[valid_idx, "user_id"])
test_users = set(df.loc[test_idx, "user_id"])

assert train_users.isdisjoint(valid_users)
assert train_users.isdisjoint(test_users)
assert valid_users.isdisjoint(test_users)

assert (
    train_users | valid_users | test_users
) == eligible_users

assert df["split"].isin(["train", "valid", "test"]).all()

# 5. 공통 분할표 저장
# 피처와 정답은 넣지 않고 고객 ID와 소속만 저장합니다.
split_manifest = df[["user_id", "split"]].copy()

split_path = (
    PROCESSED_DIR / "customer_split_20181101_v1.csv"
)

split_manifest.to_csv(
    split_path,
    index=False,
    encoding="utf-8-sig",
)

# 저장 후 다시 읽어 일치 확인
checked_split = pd.read_csv(
    split_path,
    dtype={"user_id": "string", "split": "string"},
)

pd.testing.assert_frame_equal(
    checked_split,
    split_manifest.astype({
        "user_id": "string",
        "split": "string",
    }),
)

# 6. 분할별 고객 수와 이탈 비율 출력
summary_rows = []

for split_name in ["train", "valid", "test"]:
    part = df.loc[df["split"] == split_name]

    summary_rows.append({
        "구분": split_name,
        "고객 수": len(part),
        "이탈 고객 수": int(part["churn"].sum()),
        "이탈 비율(%)": round(
            part["churn"].mean() * 100, 2
        ),
    })

split_summary = pd.DataFrame(summary_rows)

print("===== 공통 분할 생성 완료 =====")
print(split_summary.to_string(index=False))
print("\n고객 중복·누락 없음")
print("분할표 저장:", split_path)
print("기존 고객별 피처 CSV는 변경하지 않았습니다.")

===== 공통 분할 생성 완료 =====
   구분  고객 수  이탈 고객 수  이탈 비율(%)
train 12697      724      5.70
valid  2721      156      5.73
 test  2721      155      5.70

고객 중복·누락 없음
분할표 저장: D:\Study\study\Project\2nd Project\data\processed\customer_split_20181101_v1.csv
기존 고객별 피처 CSV는 변경하지 않았습니다.


In [21]:
import pandas as pd
import json

# 방금 분할한 데이터가 메모리에 있는지 확인
assert "split" in df.columns
assert df["user_id"].is_unique

assert df["split"].value_counts().to_dict() == {
    "train": 12_697,
    "valid": 2_721,
    "test": 2_721,
}

# 1. split을 포함한 통합 파일 저장
combined_path = (
    PROCESSED_DIR
    / "customer_features_20181101_v1_with_split.csv"
)

df.to_csv(
    combined_path,
    index=False,
    encoding="utf-8-sig",
)

# 2. 개별 파일 3개 저장
separate_paths = {}

for split_name in ["train", "valid", "test"]:
    part = (
        df.loc[df["split"] == split_name]
        .drop(columns="split")
        .reset_index(drop=True)
    )

    part_path = (
        PROCESSED_DIR / f"{split_name}_20181101_v1.csv"
    )

    part.to_csv(
        part_path,
        index=False,
        encoding="utf-8-sig",
    )

    separate_paths[split_name] = part_path

# 3. 통합 파일을 다시 읽어 저장 전후 비교
combined_check = pd.read_csv(
    combined_path,
    dtype={
        "user_id": "string",
        "level_last": "string",
        "split": "string",
    },
)

pd.testing.assert_frame_equal(
    combined_check,
    df.reset_index(drop=True),
    check_dtype=False,
)

# 공통 분할표와도 같은지 확인
manifest_check = pd.read_csv(
    split_path,
    dtype={"user_id": "string", "split": "string"},
)

pd.testing.assert_frame_equal(
    combined_check[["user_id", "split"]]
        .sort_values("user_id").reset_index(drop=True),
    manifest_check.sort_values("user_id").reset_index(drop=True),
    check_dtype=False,
)

# 4. 개별 파일과 통합 파일의 내용 비교
file_summary = []

for split_name, part_path in separate_paths.items():
    separate_check = pd.read_csv(
        part_path,
        dtype={
            "user_id": "string",
            "level_last": "string",
        },
    )

    expected = (
        combined_check.loc[
            combined_check["split"] == split_name
        ]
        .drop(columns="split")
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        separate_check,
        expected,
        check_dtype=False,
    )

    file_summary.append({
        "구분": split_name,
        "고객 수": len(separate_check),
        "이탈 고객 수": int(separate_check["churn"].sum()),
        "열 수": len(separate_check.columns),
    })

# 5. 저장 확인 보고서
save_report = {
    "통합 파일": str(combined_path),
    "통합 고객 수": len(combined_check),
    "통합 열 수": len(combined_check.columns),
    "개별 파일": {
        name: str(path)
        for name, path in separate_paths.items()
    },
    "개별 파일 집계": file_summary,
    "통합_개별_분할표_일치": True,
}

report_path = REPORT_DIR / "12_배포용CSV_저장확인.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        save_report,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("===== 통합·개별 CSV 저장 완료 =====")
print("통합 파일:", combined_path.name)
print(f"통합 고객 수: {len(combined_check):,}명")
print(f"통합 열 수: {len(combined_check.columns)}개")

print("\n개별 파일:")
print(pd.DataFrame(file_summary).to_string(index=False))

print("\n통합 파일·개별 파일·공통 분할표 일치 확인")
print("저장 폴더:", PROCESSED_DIR)
print("보고서:", report_path)

===== 통합·개별 CSV 저장 완료 =====
통합 파일: customer_features_20181101_v1_with_split.csv
통합 고객 수: 18,139명
통합 열 수: 17개

개별 파일:
   구분  고객 수  이탈 고객 수  열 수
train 12697      724   16
valid  2721      156   16
 test  2721      155   16

통합 파일·개별 파일·공통 분할표 일치 확인
저장 폴더: D:\Study\study\Project\2nd Project\data\processed
보고서: D:\Study\study\Project\2nd Project\reports\12_배포용CSV_저장확인.json


In [2]:
import json
from collections import Counter

# 통합 CSV와 같은 고객·분할·정답을 사용
split_by_user = dict(
    zip(combined_check["user_id"], combined_check["split"])
)

label_by_user = dict(
    zip(combined_check["user_id"], combined_check["churn"])
)

assert set(customer_stats) == set(split_by_user)

sequence_path = (
    PROCESSED_DIR / "customer_sequences_20181101_v1.jsonl"
)

# 1. 고객 한 명당 한 줄로 저장
with sequence_path.open("w", encoding="utf-8") as f:
    for user_id in sorted(customer_stats):
        recent_events = customer_stats[user_id]["recent_events"]

        # 이벤트 시각, 같은 시각이면 원본 줄 번호 순서
        assert recent_events == sorted(recent_events)
        assert 1 <= len(recent_events) <= 50

        events = [
            {
                "ts": ts,
                "page": page,
            }
            for ts, line_number, page in recent_events
        ]

        sequence_record = {
            "user_id": user_id,
            "split": split_by_user[user_id],
            "churn": int(label_by_user[user_id]),
            "sequence_length": len(events),
            "events": events,
        }

        f.write(
            json.dumps(
                sequence_record,
                ensure_ascii=False
            ) + "\n"
        )

# 2. 저장한 파일을 다시 읽어 검증
checked_users = set()
sequence_split_counts = Counter()
short_sequence_count = 0

with sequence_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        user_id = record["user_id"]
        events = record["events"]

        assert user_id not in checked_users
        checked_users.add(user_id)

        assert record["split"] == split_by_user[user_id]
        assert record["churn"] == int(label_by_user[user_id])
        assert record["sequence_length"] == len(events)
        assert 1 <= len(events) <= 50

        # 저장 전 행동 목록과 같은지 확인
        expected_events = [
            {"ts": ts, "page": page}
            for ts, line_number, page
            in customer_stats[user_id]["recent_events"]
        ]

        assert events == expected_events

        # 관찰 기간 밖의 기록이 없는지 확인
        assert all(
            OBS_START_MS <= event["ts"] < CUTOFF_MS
            for event in events
        )

        # 해지 완료가 입력 행동에 포함되지 않았는지 확인
        assert all(
            event["page"] != CHURN_EVENT
            for event in events
        )

        sequence_split_counts[record["split"]] += 1

        if len(events) < 50:
            short_sequence_count += 1

assert checked_users == set(split_by_user)

assert dict(sequence_split_counts) == (
    combined_check["split"].value_counts().to_dict()
)

# 3. 저장 확인 보고서
sequence_report = {
    "파일": str(sequence_path),
    "고객 수": len(checked_users),
    "분할별 고객 수": dict(sequence_split_counts),
    "최대 행동 수": 50,
    "행동 수가 50개 미만인 고객 수": short_sequence_count,
    "정형 데이터와 고객_분할_정답 일치": True,
    "행동 정렬 기준": "이벤트 시각, 같은 시각이면 원본 줄 번호",
    "패딩 적용": False,
    "숫자 인코딩 적용": False,
}

report_path = REPORT_DIR / "13_행동시퀀스_저장확인.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        sequence_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("===== 행동 시퀀스 저장 및 검증 완료 =====")
print("파일:", sequence_path)
print(f"고객 수: {len(checked_users):,}명")
print("분할별 고객 수:", dict(sequence_split_counts))
print(f"행동이 50개 미만인 고객: {short_sequence_count:,}명")
print("정형 데이터와 고객·분할·정답 일치")
print("관찰 기간과 저장 전후 행동 목록 일치")
print("보고서:", report_path)

NameError: name 'combined_check' is not defined

In [23]:
import json
from collections import Counter

# 통합 CSV와 같은 고객·분할·정답을 사용
split_by_user = dict(
    zip(combined_check["user_id"], combined_check["split"])
)

label_by_user = dict(
    zip(combined_check["user_id"], combined_check["churn"])
)

assert set(customer_stats) == set(split_by_user)

sequence_path = (
    PROCESSED_DIR / "customer_sequences_20181101_v1.jsonl"
)

# 1. 고객 한 명당 한 줄로 저장
with sequence_path.open("w", encoding="utf-8") as f:
    for user_id in sorted(customer_stats):
        recent_events = customer_stats[user_id]["recent_events"]

        # 이벤트 시각, 같은 시각이면 원본 줄 번호 순서
        assert recent_events == sorted(recent_events)
        assert 1 <= len(recent_events) <= 50

        events = [
            {
                "ts": ts,
                "page": page,
            }
            for ts, line_number, page in recent_events
        ]

        sequence_record = {
            "user_id": user_id,
            "split": split_by_user[user_id],
            "churn": int(label_by_user[user_id]),
            "sequence_length": len(events),
            "events": events,
        }

        f.write(
            json.dumps(
                sequence_record,
                ensure_ascii=False
            ) + "\n"
        )

# 2. 저장한 파일을 다시 읽어 검증
checked_users = set()
sequence_split_counts = Counter()
short_sequence_count = 0

with sequence_path.open("r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)

        user_id = record["user_id"]
        events = record["events"]

        assert user_id not in checked_users
        checked_users.add(user_id)

        assert record["split"] == split_by_user[user_id]
        assert record["churn"] == int(label_by_user[user_id])
        assert record["sequence_length"] == len(events)
        assert 1 <= len(events) <= 50

        # 저장 전 행동 목록과 같은지 확인
        expected_events = [
            {"ts": ts, "page": page}
            for ts, line_number, page
            in customer_stats[user_id]["recent_events"]
        ]

        assert events == expected_events

        # 관찰 기간 밖의 기록이 없는지 확인
        assert all(
            OBS_START_MS <= event["ts"] < CUTOFF_MS
            for event in events
        )

        # 해지 완료가 입력 행동에 포함되지 않았는지 확인
        assert all(
            event["page"] != CHURN_EVENT
            for event in events
        )

        sequence_split_counts[record["split"]] += 1

        if len(events) < 50:
            short_sequence_count += 1

assert checked_users == set(split_by_user)

assert dict(sequence_split_counts) == (
    combined_check["split"].value_counts().to_dict()
)

# 3. 저장 확인 보고서
sequence_report = {
    "파일": str(sequence_path),
    "고객 수": len(checked_users),
    "분할별 고객 수": dict(sequence_split_counts),
    "최대 행동 수": 50,
    "행동 수가 50개 미만인 고객 수": short_sequence_count,
    "정형 데이터와 고객_분할_정답 일치": True,
    "행동 정렬 기준": "이벤트 시각, 같은 시각이면 원본 줄 번호",
    "패딩 적용": False,
    "숫자 인코딩 적용": False,
}

report_path = REPORT_DIR / "13_행동시퀀스_저장확인.json"

with report_path.open("w", encoding="utf-8") as f:
    json.dump(
        sequence_report,
        f,
        ensure_ascii=False,
        indent=2
    )

print("===== 행동 시퀀스 저장 및 검증 완료 =====")
print("파일:", sequence_path)
print(f"고객 수: {len(checked_users):,}명")
print("분할별 고객 수:", dict(sequence_split_counts))
print(f"행동이 50개 미만인 고객: {short_sequence_count:,}명")
print("정형 데이터와 고객·분할·정답 일치")
print("관찰 기간과 저장 전후 행동 목록 일치")
print("보고서:", report_path)

===== 행동 시퀀스 저장 및 검증 완료 =====
파일: D:\Study\study\Project\2nd Project\data\processed\customer_sequences_20181101_v1.jsonl
고객 수: 18,139명
분할별 고객 수: {'train': 12697, 'valid': 2721, 'test': 2721}
행동이 50개 미만인 고객: 1,859명
정형 데이터와 고객·분할·정답 일치
관찰 기간과 저장 전후 행동 목록 일치
보고서: D:\Study\study\Project\2nd Project\reports\13_행동시퀀스_저장확인.json


In [1]:
import pandas as pd

# 저장된 통합 CSV 읽기
distribution_df = pd.read_csv(
    PROCESSED_DIR
    / "customer_features_20181101_v1_with_split.csv",
    dtype={
        "user_id": "string",
        "level_last": "string",
        "split": "string",
    },
)

# 학습 고객만 선택
train_for_check = distribution_df.loc[
    distribution_df["split"] == "train"
].copy()

assert len(train_for_check) == 12_697

numeric_features = [
    name
    for name, definition in FEATURE_DEFINITIONS.items()
    if definition["종류"] == "수치형"
]

train_numeric = train_for_check[numeric_features]

# 피처별 분포 요약
distribution_summary = pd.DataFrame({
    "최솟값": train_numeric.min(),
    "중앙값": train_numeric.median(),
    "95백분위": train_numeric.quantile(0.95),
    "99백분위": train_numeric.quantile(0.99),
    "최댓값": train_numeric.max(),
    "0인 비율(%)": train_numeric.eq(0).mean() * 100,
})

distribution_summary.index.name = "피처"

summary_path = (
    REPORT_DIR / "14_학습데이터_수치분포.csv"
)

distribution_summary.to_csv(
    summary_path,
    encoding="utf-8-sig",
)

print("===== 학습 데이터 수치 분포 =====")
print(f"확인한 고객 수: {len(train_for_check):,}명")

print(
    distribution_summary.to_string(
        float_format=lambda value: f"{value:.3f}"
    )
)

print("\n보고서 저장:", summary_path)
print("고객 삭제·값 수정·스케일링은 적용하지 않았습니다.")

NameError: name 'PROCESSED_DIR' is not defined

In [3]:
from pathlib import Path
import json

PROJECT_DIR = Path(r"D:\Study\study\Project\2nd Project")

RAW_PATH = PROJECT_DIR / "data" / "row" / "data.json"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
REPORT_DIR = PROJECT_DIR / "reports"

# 저장해둔 피처 정의서에서 복원
definition_path = REPORT_DIR / "10_피처_정의서.json"

with definition_path.open("r", encoding="utf-8") as f:
    feature_spec = json.load(f)

FEATURE_DEFINITIONS = feature_spec["피처 정의"]

# 이번 작업에 필요한 파일 확인
combined_path = (
    PROCESSED_DIR
    / "customer_features_20181101_v1_with_split.csv"
)

assert combined_path.is_file(), (
    f"통합 CSV를 찾을 수 없습니다: {combined_path}"
)

print("변수 복원 완료")
print("통합 CSV 존재:", combined_path.is_file())
print("입력 피처 수:", len(FEATURE_DEFINITIONS))

변수 복원 완료
통합 CSV 존재: True
입력 피처 수: 14


In [4]:
import pandas as pd

# 저장된 통합 CSV 읽기
distribution_df = pd.read_csv(
    PROCESSED_DIR
    / "customer_features_20181101_v1_with_split.csv",
    dtype={
        "user_id": "string",
        "level_last": "string",
        "split": "string",
    },
)

# 학습 고객만 선택
train_for_check = distribution_df.loc[
    distribution_df["split"] == "train"
].copy()

assert len(train_for_check) == 12_697

numeric_features = [
    name
    for name, definition in FEATURE_DEFINITIONS.items()
    if definition["종류"] == "수치형"
]

train_numeric = train_for_check[numeric_features]

# 피처별 분포 요약
distribution_summary = pd.DataFrame({
    "최솟값": train_numeric.min(),
    "중앙값": train_numeric.median(),
    "95백분위": train_numeric.quantile(0.95),
    "99백분위": train_numeric.quantile(0.99),
    "최댓값": train_numeric.max(),
    "0인 비율(%)": train_numeric.eq(0).mean() * 100,
})

distribution_summary.index.name = "피처"

summary_path = (
    REPORT_DIR / "14_학습데이터_수치분포.csv"
)

distribution_summary.to_csv(
    summary_path,
    encoding="utf-8-sig",
)

print("===== 학습 데이터 수치 분포 =====")
print(f"확인한 고객 수: {len(train_for_check):,}명")

print(
    distribution_summary.to_string(
        float_format=lambda value: f"{value:.3f}"
    )
)

print("\n보고서 저장:", summary_path)
print("고객 삭제·값 수정·스케일링은 적용하지 않았습니다.")

===== 학습 데이터 수치 분포 =====
확인한 고객 수: 12,697명
                               최솟값     중앙값    95백분위    99백분위      최댓값  0인 비율(%)
피처                                                                             
active_days                  1.000   6.000   19.000   24.000   28.000     0.000
session_count                1.000   5.000   21.000   34.000   70.000     0.000
song_count                   0.000 267.000 1732.000 2827.120 5731.000     0.244
days_since_last_activity     0.000   2.490   19.329   26.037   27.974     0.000
recent_7d_song_count         0.000  50.000  556.000  903.040 1584.000    27.857
previous_7d_song_count       0.000  49.000  534.000  894.000 1803.000    27.605
song_count_change        -1233.000   0.000  323.000  592.000 1113.000    11.081
thumbs_up_count              0.000  14.000   99.000  167.000  359.000     4.970
thumbs_down_count            0.000   3.000   20.000   33.000   70.000    19.469
playlist_add_count           0.000   8.000   51.000   82.040  166.000     9.2

In [5]:
import json

numeric_policy = {
    "검토 자료": "14_학습데이터_수치분포.csv",
    "분포 확인 대상": "학습 고객 12,697명",

    "처리 기준": {
        "큰 이용량만을 근거로 고객 삭제": False,
        "상위 백분위 기준으로 값 잘라내기": False,
        "0인 행동 횟수": "행동이 없다는 정보로 유지",
        "음수 이용량 변화": "이용량 감소 정보로 유지",
        "현재 CSV의 수치 변경": False,
    },

    "판단 근거": (
        "자료형과 기본 범위 검사를 통과한 값에 대해 "
        "분포만으로 오류나 봇이라고 판단할 근거가 부족함"
    ),

    "모델 학습 단계 지침": {
        "로지스틱 회귀 및 MLP": (
            "학습 데이터로 스케일러를 학습하고 "
            "검증·최종 평가·앱 입력에는 같은 변환을 적용"
        ),
        "LightGBM": (
            "일반적으로 수치형 스케일링 없이 학습 가능"
        ),
        "범주형 level_last": (
            "모델에 맞는 인코딩을 적용하고 "
            "인코더와 범주 기준을 모델과 함께 저장"
        ),
        "금지 사항": (
            "전체 데이터로 스케일링 기준을 계산하거나 "
            "최종 평가 성능을 보고 처리 기준을 변경하지 않음"
        ),
    },

    "현재 데이터 상태": (
        "스케일링과 범주형 인코딩을 적용하지 않은 고객별 피처"
    ),
}

policy_path = REPORT_DIR / "15_수치형피처_처리기준.json"

with policy_path.open("w", encoding="utf-8") as f:
    json.dump(
        numeric_policy,
        f,
        ensure_ascii=False,
        indent=2
    )

print("수치형 피처 처리 기준 저장 완료")
print("큰 값·0·음수 변화량은 의미에 맞게 보존합니다.")
print("고객 삭제와 값 잘라내기는 적용하지 않았습니다.")
print("스케일링·인코딩은 모델 학습 단계에서 적용합니다.")
print("저장 위치:", policy_path)

수치형 피처 처리 기준 저장 완료
큰 값·0·음수 변화량은 의미에 맞게 보존합니다.
고객 삭제와 값 잘라내기는 적용하지 않았습니다.
스케일링·인코딩은 모델 학습 단계에서 적용합니다.
저장 위치: D:\Study\study\Project\2nd Project\reports\15_수치형피처_처리기준.json


In [6]:
import json
import pandas as pd

# 저장된 데이터와 피처 정의서를 기준으로 작성
data = pd.read_csv(
    PROCESSED_DIR
    / "customer_features_20181101_v1_with_split.csv",
    dtype={
        "user_id": "string",
        "level_last": "string",
        "split": "string",
    },
)

with (
    REPORT_DIR / "10_피처_정의서.json"
).open("r", encoding="utf-8") as f:
    feature_spec = json.load(f)

definitions = feature_spec["피처 정의"]

# 피처 설명 표
feature_table = [
    "| 피처 | 종류 | 의미 |",
    "|---|---|---|",
]

for name, info in definitions.items():
    feature_table.append(
        f"| {name} | {info['종류']} | {info['설명']} |"
    )

# 실제 분할 결과 표
split_table = [
    "| 구분 | 고객 수 | 이탈 고객 수 |",
    "|---|---:|---:|",
]

for split_name in ["train", "valid", "test"]:
    part = data.loc[data["split"] == split_name]

    split_table.append(
        f"| {split_name} | {len(part):,} | "
        f"{int(part['churn'].sum()):,} |"
    )

readme = f"""# 팀 공통 데이터 사용 설명서

## 1. 예측 문제

최근 28일의 이용 행동으로 이후 14일 안의 서비스 해지를 예측합니다.
무료·유료 이용 상태의 가입 고객을 포함하며, 요금제 하향과 서비스 해지는 구분합니다.

- 시간대: UTC
- 관찰 기간: 2018-10-04 00:00 이상 ~ 2018-11-01 00:00 미만
- 예측 기간: 2018-11-01 00:00 이상 ~ 2018-11-15 00:00 미만
- 대상: 관찰 기간에 활동했고, 기준일 전에 해지 완료가 확인되지 않은 고객
- churn = 1: 예측 기간에 Cancellation Confirmation이 확인됨
- churn = 0: 예측 기간에 해당 사건이 확인되지 않음
- 정답 0은 영구 유지 고객이라는 뜻이 아닙니다.

## 2. 데이터 규모

- 전체 고객: {len(data):,}명
- 이탈 고객: {int(data['churn'].sum()):,}명
- 이탈 비율: {data['churn'].mean():.2%}
- 입력 피처: {len(definitions)}개

{chr(10).join(split_table)}

## 3. 파일 선택 방법

다음 두 방식 중 편한 것을 선택합니다. 고객과 값은 동일합니다.

### 통합 파일 사용
customer_features_20181101_v1_with_split.csv

split 열의 train, valid, test 값으로 구분합니다.

### 개별 파일 사용
- train_20181101_v1.csv
- valid_20181101_v1.csv
- test_20181101_v1.csv

### 보조 파일
- customer_split_20181101_v1.csv: 고객 ID와 공통 분할 명단
- customer_sequences_20181101_v1.jsonl: DL용 최근 행동 목록
- customer_features_20181101_v1.csv: 분할 표시가 없는 전체 피처 표

모델 비교에는 통합 분할 파일 또는 개별 파일을 사용합니다.
분할 표시가 없는 전체 피처 표를 각자 다시 나누지 않습니다.

## 4. 모델 입력과 정답 구분

- 입력: 아래 피처 14개
- 정답: churn
- 입력에서 제외: user_id, churn, split
- user_id는 문자열로 읽습니다.

{chr(10).join(feature_table)}

## 5. 처리한 내용

- 가입 정보가 없고 모든 기록이 비로그인·게스트 상태인 ID 1261737의 기록 제외
- 가입 시각과 이벤트 시각의 모순을 확인하여 가입 시각 기반 피처 제외
- 가입 시각 문제만을 근거로 고객의 행동·해지 기록을 삭제하지 않음
- 모든 입력 피처와 행동 시퀀스는 관찰 기간의 기록으로만 생성
- 고객별 피처의 자료형, 기본 범위, 계산식과 정답 수 검사
- 저장 전후 값과 통합·개별 파일·분할표의 일치 확인
- 시퀀스의 고객·분할·정답·기간과 저장 전후 내용 확인
- 학습 고객의 수치 분포를 검토하고 큰 값만을 근거로 삭제하거나 자르지 않음

검사 결과는 reports 폴더에 있습니다.
위 검사는 원본의 모든 의미적 오류나 전체 완전 중복을 검증했다는 뜻은 아닙니다.

## 6. 학습 담당자 주의사항

- 공통 고객 분할을 유지합니다.
- 스케일링과 범주형 인코딩은 아직 적용하지 않았습니다.
- 필요한 전처리 기준은 학습 데이터로만 학습합니다.
- 검증·최종 평가·앱 입력에는 같은 전처리를 적용합니다.
- level_last는 free 또는 paid인 범주형 변수입니다.
- 0인 행동 횟수와 음수 song_count_change를 임의로 결측이나 오류로 바꾸지 않습니다.
- 최종 평가 데이터로 튜닝하거나 판정 임계값을 선택하지 않습니다.
- 전처리기, 피처 순서와 모델 설정을 모델과 함께 저장합니다.

## 7. DL 행동 시퀀스 사용

- 고객당 한 줄인 JSONL 형식입니다.
- user_id, split, churn, sequence_length, events가 있습니다.
- events는 관찰 기간의 최근 최대 50개 행동이며 시간순입니다.
- 같은 시각의 행동은 원본 줄 순서로 정렬했습니다.
- 50개보다 짧은 기록을 반복해서 채우지 않았습니다.
- 패딩과 숫자 인코딩은 아직 적용하지 않았습니다.
- 행동 사전은 학습 고객의 기록으로 만들고 미지 행동 처리 기준을 둡니다.
- 패딩과 실제 길이를 구분하여 모델에서 처리합니다.
- 정형 데이터와 연결할 때 행 순서가 아닌 user_id를 사용합니다.
- churn과 split은 입력 행동에 포함하지 않습니다.

## 8. 해석과 발표 범위

- NextSong 횟수는 완청 횟수나 실제 청취 시간이 아닙니다.
- SHAP 등은 모델 판단의 설명이며 실제 해지 원인의 증명이 아닙니다.
- 이번 분할은 동일 기준일의 다른 고객에 대한 평가입니다.
- 다른 달에서도 같은 성능을 낸다고 보장하지 않습니다.
- 데이터 제공자가 명시한 출처는 Udacity입니다.
- 생성 방식과 모의 데이터 여부는 별도 출처 확인 결과를 발표에 반영해야 합니다.
"""

readme_path = PROCESSED_DIR / "README_DATA.md"

readme_path.write_text(readme, encoding="utf-8")

# 저장 내용 재확인
assert readme_path.read_text(encoding="utf-8") == readme

print("팀원용 데이터 사용 설명서 저장 완료")
print("피처 설명:", len(definitions), "개")
print("저장 위치:", readme_path)
print("기존 데이터 파일은 변경하지 않았습니다.")

팀원용 데이터 사용 설명서 저장 완료
피처 설명: 14 개
저장 위치: D:\Study\study\Project\2nd Project\data\processed\README_DATA.md
기존 데이터 파일은 변경하지 않았습니다.


In [7]:
from pathlib import Path
import hashlib
import json
import zipfile

# 전달할 데이터와 설명서
data_names = [
    "customer_features_20181101_v1.csv",
    "customer_features_20181101_v1_with_split.csv",
    "train_20181101_v1.csv",
    "valid_20181101_v1.csv",
    "test_20181101_v1.csv",
    "customer_split_20181101_v1.csv",
    "customer_sequences_20181101_v1.jsonl",
    "README_DATA.md",
]

report_names = [
    "01_원본_구조_자료형_검사.json",
    "02_누락_발생상황_검사.json",
    "03_비로그인_공통ID_점검.json",
    "04_고객분석_제외기준.json",
    "05_숫자범위_범주값_검사.json",
    "06_가입시각_정합성_검사.json",
    "07_가입시각_처리기준.json",
    "08_분석기간_설정.json",
    "09_분석대상_정답라벨.json",
    "10_피처_정의서.json",
    "11_고객별CSV_저장확인.json",
    "12_배포용CSV_저장확인.json",
    "13_행동시퀀스_저장확인.json",
    "14_학습데이터_수치분포.csv",
    "15_수치형피처_처리기준.json",
]

notebook_path = (
    PROJECT_DIR / "01_데이터검증_및_전처리.ipynb"
)

files_to_pack = (
    [PROCESSED_DIR / name for name in data_names]
    + [REPORT_DIR / name for name in report_names]
    + [notebook_path]
)

# 누락 파일이 있으면 ZIP을 만들기 전에 중단
missing_files = [
    str(path) for path in files_to_pack
    if not path.is_file()
]

if missing_files:
    raise FileNotFoundError(
        "다음 파일의 위치나 이름을 확인해주세요:\n"
        + "\n".join(missing_files)
    )


def file_sha256(path):
    """파일 내용 확인용 해시를 계산합니다."""
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


# 전달 파일 목록과 내용 확인용 해시
manifest = []

for path in files_to_pack:
    manifest.append({
        "파일": path.relative_to(PROJECT_DIR).as_posix(),
        "크기_bytes": path.stat().st_size,
        "sha256": file_sha256(path),
    })

delivery_dir = PROJECT_DIR / "deliverables"
delivery_dir.mkdir(parents=True, exist_ok=True)

zip_path = delivery_dir / "music_churn_data_v1.zip"

with zipfile.ZipFile(
    zip_path,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in files_to_pack:
        archive.write(
            path,
            arcname=path.relative_to(PROJECT_DIR).as_posix(),
        )

    archive.writestr(
        "파일목록_및_해시.json",
        json.dumps(
            manifest,
            ensure_ascii=False,
            indent=2,
        ).encode("utf-8"),
    )

# ZIP 손상 및 포함 파일 목록 확인
with zipfile.ZipFile(zip_path, "r") as archive:
    damaged_file = archive.testzip()
    assert damaged_file is None, (
        f"압축 파일 검사 실패: {damaged_file}"
    )

    expected_names = {
        item["파일"] for item in manifest
    } | {"파일목록_및_해시.json"}

    assert set(archive.namelist()) == expected_names

print("===== 팀원 전달용 ZIP 생성 완료 =====")
print("저장 위치:", zip_path)
print("포함한 자료 수:", len(files_to_pack), "개")
print(
    "ZIP 크기:",
    round(zip_path.stat().st_size / (1024 ** 2), 2),
    "MiB"
)
print("압축 파일 손상·포함 목록 검사 통과")
print("12GB 원본은 포함하지 않았습니다.")

FileNotFoundError: 다음 파일의 위치나 이름을 확인해주세요:
D:\Study\study\Project\2nd Project\01_데이터검증_및_전처리.ipynb